# Catalyst Optimizer Internals

## What is Catalyst?

Catalyst is the query optimizer at the heart of Spark SQL. Every DataFrame operation — whether written in Python, Scala, R, or raw SQL — is compiled into a **tree of logical operations** and then handed to Catalyst, which rewrites that tree into the most efficient physical execution plan it can find.

Catalyst is not a traditional rule-based optimizer (like Volcano/Cascades). It is written in Scala using **functional tree transformations** — a pattern that makes it straightforward to add new rules without breaking existing ones. Each rule is simply a function: `LogicalPlan => LogicalPlan`.

### Why this matters to you as an engineer

Understanding Catalyst lets you:
- Write queries that *assist* the optimizer (filter early, project only needed columns).
- Diagnose slow jobs by reading `explain()` output confidently.
- Know when Catalyst will help you (predicate pushdown, constant folding) and when it won't (UDFs are opaque to Catalyst).
- Make informed decisions about broadcast hints, repartitioning, and caching.

## The Four Phases of Catalyst

```
  Your code (DataFrame API / SQL string)
          │
          ▼
  ┌─────────────────────────┐
  │  1. PARSED LOGICAL PLAN │  Unresolved tree — column names are just strings,
  │     (Unresolved)        │  no type information, no catalog lookup yet.
  └────────────┬────────────┘
               │  Analyzer (resolves names against catalog)
               ▼
  ┌─────────────────────────┐
  │  2. ANALYZED LOGICAL    │  Column references resolved to actual Attributes
  │     PLAN                │  with data types. Missing columns surface here
  │     (Resolved)          │  as AnalysisExceptions.
  └────────────┬────────────┘
               │  Optimizer (batch of rewrite rules — run to fixed point)
               ▼
  ┌─────────────────────────┐
  │  3. OPTIMIZED LOGICAL   │  Predicates pushed down, constants folded,
  │     PLAN                │  redundant projections eliminated, filters
  │                         │  combined (conjunctive predicate merging).
  └────────────┬────────────┘
               │  Planner (enumerates physical strategies)
               ▼
  ┌─────────────────────────┐
  │  4. PHYSICAL PLAN(S)    │  One or more candidate plans, e.g.:
  │                         │  BroadcastHashJoin vs SortMergeJoin
  └────────────┬────────────┘
               │  Cost model (CBO if statistics available; else heuristics)
               ▼
  ┌─────────────────────────┐
  │  SELECTED PHYSICAL PLAN │  Annotated with codegen stages (WholeStageCodegen)
  │  + Tungsten Codegen     │  and adaptive query execution adjustments.
  └─────────────────────────┘
               │
               ▼
          RDD execution
```

You can inspect every level with `explain(mode)` where `mode` is one of:
- `"simple"` — just the physical plan (default)
- `"extended"` — all four plans
- `"codegen"` — the generated Java source code
- `"cost"` — physical plan + cost statistics
- `"formatted"` — structured output, easiest to read for complex plans

In [ ]:
// ─────────────────────────────────────────────────────────────────────────────
// Almond (Scala kernel for Jupyter) bootstraps a SparkSession automatically
// when you import the implicit conversions. However, to target our cluster
// and set custom configs we create it explicitly.
// ─────────────────────────────────────────────────────────────────────────────
import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("CatalystOptimizerInternals")
  .master("spark://spark-master:7077")
  // Keep shuffle partitions low for demo — the default 200 creates unnecessary overhead
  .config("spark.sql.shuffle.partitions", "4")
  // Enable Cost-Based Optimizer (CBO) so Catalyst can use table statistics
  // when choosing between join strategies.
  .config("spark.sql.cbo.enabled", "true")
  .config("spark.sql.cbo.joinReorder.enabled", "true")
  .getOrCreate()

import spark.implicits._  // brings .toDF(), dataset encoders, etc. into scope
spark.sparkContext.setLogLevel("WARN")
println(s"Spark version: ${spark.version}")

// ── Sample data: orders ──────────────────────────────────────────────────────
// Seq of tuples → DataFrame via Spark implicit conversion
val ordersData = Seq(
  (1, 101, 250.0, "2024-01-15"),
  (2, 102,  80.0, "2024-01-16"),
  (3, 101, 430.0, "2024-01-17"),
  (4, 103,  15.0, "2024-01-17"),
  (5, 104, 999.0, "2024-01-18"),
  (6, 102, 120.0, "2024-01-18"),
  (7, 105,  45.0, "2024-01-19")
)
val orders: DataFrame = ordersData
  .toDF("orderId", "customerId", "amount", "orderDate")

// ── Sample data: customers ───────────────────────────────────────────────────
val customersData = Seq(
  (101, "Alice",   "US"),
  (102, "Bob",     "UK"),
  (103, "Charlie", "US"),
  (104, "Diana",   "DE"),
  (105, "Evan",    "UK")
)
val customers: DataFrame = customersData
  .toDF("customerId", "name", "country")

// Register as temp views so we can also query them with spark.sql()
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

println("DataFrames created. Schema:")
orders.printSchema()
customers.printSchema()

In [ ]:
// ─────────────────────────────────────────────────────────────────────────────
// explain("extended") emits all four plan levels.
// Read it bottom-up: the physical plan at the top drives execution;
// the parsed plan at the bottom is closest to your source code.
// ─────────────────────────────────────────────────────────────────────────────

val joinQuery: DataFrame = orders
  .join(customers, Seq("customerId"), "inner")  // equi-join on customerId
  .filter(col("amount") > 100)                  // filter AFTER join (we'll fix this next cell)
  .select("orderId", "name", "country", "amount")

println("=" * 80)
println("FULL EXPLAIN (extended) — all 4 plan levels")
println("=" * 80)
joinQuery.explain("extended")
// ── How to read the output ──────────────────────────────────────────────────
// == Parsed Logical Plan ==
//   Column names are UnresolvedAttribute nodes — the catalog hasn't been
//   consulted yet. 'amount' is just a string here.
//
// == Analyzed Logical Plan ==
//   Now 'amount' is resolved to orders.amount#5: double. Types are attached.
//   Any typo in a column name would have caused an AnalysisException here.
//
// == Optimized Logical Plan ==
//   Key changes to look for:
//   - Filter pushed down (below the Join node)
//   - Unused columns dropped before the join (column pruning)
//
// == Physical Plan ==
//   Shows concrete operators: BroadcastHashJoin (for small tables) or
//   SortMergeJoin (for large tables). WholeStageCodegen markers wrap
//   multiple operators that Tungsten will fuse into one JVM method.

In [ ]:
// ─────────────────────────────────────────────────────────────────────────────
// PREDICATE PUSHDOWN DEMO
// ─────────────────────────────────────────────────────────────────────────────
// A common beginner mistake: filter AFTER the join.
// This forces Spark to process ALL rows through the join before discarding.
// Catalyst fixes it automatically — but understanding when it *can't* fix it
// (e.g. on UDFs) is essential.

// ── Version A: naively filter AFTER join ────────────────────────────────────
val filterAfterJoin: DataFrame = orders
  .join(customers, Seq("customerId"), "inner")
  .filter(col("amount") > 200)   // applied to the joined result

println("Version A — filter after join (as written):")
filterAfterJoin.explain("extended")

// ── Version B: manually filter BEFORE join ──────────────────────────────────
// We explicitly push the filter to reduce the orders table first.
val filterBeforeJoin: DataFrame = orders
  .filter(col("amount") > 200)   // reduce orders BEFORE the join
  .join(customers, Seq("customerId"), "inner")

println("Version B — filter before join (explicit):")
filterBeforeJoin.explain("extended")

// ── What to notice ───────────────────────────────────────────────────────────
// In the OPTIMIZED LOGICAL PLAN for Version A you will see:
//   Filter (amount#5 > 200.0)
//   +- Join Inner
//      +- ...
// ... wait, isn't the filter ABOVE the join? Actually Catalyst moves it BELOW:
//   Join Inner
//   :- Filter (amount#5 > 200.0)   ← pushed down to the orders scan side!
//   :- ...
// Both Version A and B produce IDENTICAL optimized plans.
// Catalyst's PushDownPredicate rule fires before physical planning.

In [ ]:
// ─────────────────────────────────────────────────────────────────────────────
// CONSTANT FOLDING
// ─────────────────────────────────────────────────────────────────────────────
// Catalyst evaluates constant sub-expressions at compile time.
// It also removes predicates that are trivially true or that simplify to
// a shorter form.

// Write an intentionally redundant filter:
//   1=1       → always true  → eliminated entirely
//   true      → always true  → eliminated entirely
//   2+3 > 4   → 5 > 4        → true   → eliminated
//   amount > 0 stays — it's a runtime predicate
val withRedundantPredicates: DataFrame = orders
  .filter(
    lit(1) === lit(1) &&     // 1 = 1  (always true)
    lit(true) &&             // TRUE   (always true)
    (lit(2) + lit(3)) > 4 && // 5 > 4  (always true — constant arithmetic)
    col("amount") > 0        // real predicate — kept
  )

println("Constant folding — verify only 'amount > 0' survives in optimized plan:")
withRedundantPredicates.explain("extended")
// Expected in Optimized Logical Plan:
//   Filter (amount#5 > 0.0)
//   +- Relation[...] (no mention of the constant predicates)
//
// The ConstantFolding and BooleanSimplification rules in Catalyst's Optimizer
// batch eliminate everything that can be resolved at plan-compilation time.

In [ ]:
// ─────────────────────────────────────────────────────────────────────────────
// COLUMN PRUNING (Projection Pushdown)
// ─────────────────────────────────────────────────────────────────────────────
// When you SELECT only a subset of columns, Catalyst pushes the projection
// all the way down to the scan operator. For columnar formats like Parquet,
// this means entire column chunks are skipped on disk — massive I/O savings.

// First, write the DataFrames to Parquet so we can demonstrate a real scan
val dataPath = "/home/jovyan/data/catalyst_demo"
orders.write.mode("overwrite").parquet(s"$dataPath/orders")
customers.write.mode("overwrite").parquet(s"$dataPath/customers")

// Read back from Parquet (so the scan node is a FileScan, not an in-memory relation)
val ordersParquet    = spark.read.parquet(s"$dataPath/orders")
val customersParquet = spark.read.parquet(s"$dataPath/customers")

// Join ALL columns, then select only two
val prunedQuery: DataFrame = ordersParquet
  .join(customersParquet, Seq("customerId"), "inner")
  .select(
    col("name"),    // from customers
    col("amount")   // from orders
    // All other columns (orderId, orderDate, country, customerId) are NOT selected
  )

println("Column pruning — only 'name' and 'amount' should appear in FileScan nodes:")
prunedQuery.explain("formatted")
// In the formatted plan, look for:
//   FileScan parquet [...] PushedFilters: [], ReadSchema: struct<customerId:int,amount:double>
//   FileScan parquet [...] PushedFilters: [], ReadSchema: struct<customerId:int,name:string>
// Only the columns actually needed (plus the join key) are in the ReadSchema.
// Parquet's columnar layout means 'orderDate', 'country', etc. are never read from disk.

## Tungsten — Binary Off-Heap Execution Engine

Catalyst handles *logical* optimization. Tungsten handles *physical* execution efficiency. The two are complementary layers.

### The Problem with JVM Objects

A standard Java `String` object carrying the value `"hello"` consumes approximately **48 bytes** of heap memory:
- 16-byte object header
- 4-byte `char[]` reference
- 16-byte `char[]` header
- 10 bytes of actual characters (`char` is 2 bytes in Java)
- 2 bytes padding

The raw string data is only 5 bytes. The JVM object wrapper is **9.6× overhead**.

At Spark scale — billions of records — this heap pressure causes constant GC pauses that stall all executor threads.

### What Tungsten Does

| Feature | Description |
|---------|-------------|
| **Binary in-memory format** | Rows are stored as compact byte arrays in off-heap (or on-heap) memory. No Java objects, no GC pressure. |
| **Unsafe memory access** | Tungsten uses `sun.misc.Unsafe` to read/write raw memory — pointer arithmetic just like C. |
| **Whole-Stage Code Generation (WSCG)** | Multiple logical operators (Filter + Project + HashAggregate) are fused into a single JVM method. The JIT compiler can optimize this monolithic loop far better than a loop-per-operator design. |
| **Cache-friendly memory layout** | Column data is stored contiguously. Sequential access patterns are L1/L2-cache friendly. |

### Where you see it

In `explain()` output, operators wrapped in `*(n) ...` are inside a **WholeStageCodegen** block. The `n` is the codegen stage index. Operators *outside* these wrappers (often non-code-gen-able operators like `HashAggregate` across multiple stages) are executed in interpreted mode.

```
*(2) HashAggregate(keys=[window#0, sensorId#1], functions=[count(1)])
+- Exchange hashpartitioning(window#0, sensorId#1, 4)
   +- *(1) HashAggregate(keys=[window#0, sensorId#1], functions=[partial_count(1)])
      +- *(1) Filter (isnotnull(eventTime#2) ...)
         +- StreamingRelation
```

The `*(1)` and `*(2)` mark two fused codegen stages. The `Exchange` between them is a shuffle — codegen cannot cross a shuffle boundary.

In [ ]:
// Clean shutdown — always stop the SparkSession at the end of a notebook
// to release cluster executor slots for other jobs.
spark.stop()
println("SparkSession stopped.")